# NB01 -- Baselines Fortes: B1, B2, B3 (BTCS)
**Budgeted Temporal Case Segmentation for AML Transaction Monitoring**
Andre da Costa Silva | ITA | 2026

- **B1 Temporal-WCC**: WCC com restricao temporal -- quebra componentes com gap > delta dias
- **B2 WCC+Ctx**: WCC + adiciona arestas do periodo completo dentro do intervalo do caso
- **B3 Hub-Pruned**: WCC apos remover arestas com nos hub (grau > percentil H)

**Pre-requisito**: nb00 executado com sucesso (erro < 1% vs ICEIS)

## Celula 1 -- Setup

In [ ]:
# CELULA 1 -- Setup completo
import os, sys, subprocess, time, math, contextlib
from pathlib import Path

def _pip(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + list(pkgs))

_pip('numpy', 'pandas', 'scikit-learn', 'matplotlib', 'networkx', 'psutil', 'tqdm')

try:
    import torch
except ImportError:
    _pip('torch', 'torchvision', 'torchaudio', '--index-url', 'https://download.pytorch.org/whl/cu121')
    import torch

import numpy as np
import pandas as pd
import psutil
import matplotlib.pyplot as plt
from IPython.display import display

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'torch {torch.__version__} | device: {DEVICE}')

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
    except Exception as e:
        print(f'[WARN] Drive: {e}')

BASE = Path('/content/drive/MyDrive') if IN_COLAB else Path('.').resolve()

# Caminhos confirmados em marco/2026
AML100K_BASE  = BASE / 'DatasetDissertacao/IBM_TRANSACTION_AML/AMLSIMFULL/AML100k'
AML100K_ARTIF = AML100K_BASE / 'artifacts'
AML100K_PROBS = AML100K_BASE / 'results/probs_v4'
AML100K_MODEL = 'SAGE'
AML100K_SEED  = 42

AML1M_BASE    = BASE / 'DatasetDissertacao/IBM_TRANSACTION_AML/AMLSIMFULL/AML1M'
AML1M_ARTIF   = AML1M_BASE / 'artifacts'
AML1M_PROBS   = AML1M_BASE / 'results_aml1m_graphsage_only/probs_v4'
AML1M_MODEL   = 'GraphSAGE'
AML1M_SEED    = 44

BTCS_OUTPUT_DIR = BASE / 'GrafosGNN/results/nb01_baselines'
BTCS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('\n=== Verificacao de arquivos ===')
checks = [
    ('AML100k cache', AML100K_ARTIF / 'edge_data_v4_clean.pt'),
    ('AML100k probs', AML100K_PROBS / f'{AML100K_MODEL}_seed{AML100K_SEED}_test.npz'),
    ('AML1M   cache', AML1M_ARTIF   / 'edge_data_v4_clean.pt'),
    ('AML1M   probs', AML1M_PROBS   / f'{AML1M_MODEL}_seed{AML1M_SEED}_test.npz'),
]
for label, path in checks:
    print(f"  {'ok' if path.exists() else 'MISSING'}  {label}: {path}")

## Celula 2 -- Funcoes base

In [ ]:
# CELULA 2 -- Funcoes base: Union-Find, WCC, evaluate_cases, load_dataset_artifacts

@contextlib.contextmanager
def measure_resources():
    proc = psutil.Process(os.getpid())
    mem0 = proc.memory_info().rss / 1024**2
    t0   = time.perf_counter()
    res  = {}
    yield res
    res['time_s'] = time.perf_counter() - t0
    res['ram_mb'] = proc.memory_info().rss / 1024**2 - mem0

def _uf_init(n):
    return np.arange(n, dtype=np.int64), np.zeros(n, dtype=np.int64)

def _find(parent, a):
    while parent[a] != a:
        parent[a] = parent[parent[a]]
        a = parent[a]
    return a

def _union(parent, rank, a, b):
    ra, rb = _find(parent, a), _find(parent, b)
    if ra == rb: return
    if rank[ra] < rank[rb]: parent[ra] = rb
    elif rank[ra] > rank[rb]: parent[rb] = ra
    else: parent[rb] = ra; rank[ra] += 1

def _build_wcc(src_sel, dst_sel, sel_idx, full_eval_edges, min_nodes=3):
    nodes_all = np.unique(np.concatenate([src_sel, dst_sel]))
    node_map  = {int(n): i for i, n in enumerate(nodes_all)}
    N = len(nodes_all)
    us = np.fromiter((node_map[int(x)] for x in src_sel), dtype=np.int64)
    vs = np.fromiter((node_map[int(x)] for x in dst_sel), dtype=np.int64)
    parent, rank = _uf_init(N)
    for a, b in zip(us, vs):
        _union(parent, rank, int(a), int(b))
    comp = np.array([_find(parent, i) for i in range(N)], dtype=np.int64)
    _, comp_ids = np.unique(comp, return_inverse=True)
    sizes  = np.bincount(comp_ids)
    valid  = np.where(sizes >= min_nodes)[0]
    gid_remap = {int(c): i for i, c in enumerate(valid)}
    max_node  = int(nodes_all.max()) + 1
    gid_of    = -np.ones(max_node, dtype=np.int64)
    for node_id, local_idx in node_map.items():
        c = int(comp_ids[local_idx])
        if c in gid_remap:
            gid_of[node_id] = gid_remap[c]
    n_cases = len(valid)
    if n_cases == 0:
        return []
    cases = [{'nodes': set(), 'seed_edges': [], 'induced_edges': []} for _ in range(n_cases)]
    for i, (u, v) in enumerate(zip(src_sel, dst_sel)):
        g = gid_of[int(u)] if int(u) < max_node else -1
        if g >= 0:
            cases[g]['seed_edges'].append(int(sel_idx[i]))
            cases[g]['nodes'].update([int(u), int(v)])
    src_full = np.asarray(full_eval_edges['src'], dtype=np.int64)
    dst_full = np.asarray(full_eval_edges['dst'], dtype=np.int64)
    for i, (u, v) in enumerate(zip(src_full, dst_full)):
        gu = gid_of[int(u)] if int(u) < max_node else -1
        gv = gid_of[int(v)] if int(v) < max_node else -1
        if gu >= 0 and gu == gv:
            cases[gu]['induced_edges'].append(i)
    return cases

def evaluate_cases(cases, ground_truth, ranked_edges, k):
    y_full = np.asarray(ground_truth['y_full'], dtype=int)
    y_test = np.asarray(ranked_edges['y'],      dtype=int)
    E_test = len(y_test)
    pos_te = int(y_test.sum())
    K      = max(1, int(math.ceil(k * E_test)))
    if not cases:
        return {m: np.nan for m in ['n_cases','coverage','purity_induced','cr_at_k',
                'edges_per_case_median','edges_per_case_p95','edges_per_case_max',
                'e_ind_over_ek','ocr_b100','ocr_b500','ocr_b1000']}
    ind_sizes = np.array([len(c['induced_edges']) for c in cases])
    pos_ind   = sum(int(y_full[c['induced_edges']].sum()) for c in cases)
    coverage  = float(pos_ind / max(pos_te, 1))
    purity    = float(pos_ind / max(ind_sizes.sum(), 1))
    pos_sel   = sum(int(y_test[c['seed_edges']].sum()) for c in cases)
    recall    = float(pos_sel / max(pos_te, 1))
    cr_at_k   = float(recall / max(coverage, 1e-12)) if coverage > 0 else np.nan
    return {
        'n_cases':               len(cases),
        'coverage':              coverage,
        'purity_induced':        purity,
        'cr_at_k':               cr_at_k,
        'recall_in_cases':       recall,
        'edges_per_case_median': float(np.median(ind_sizes)),
        'edges_per_case_p95':    float(np.percentile(ind_sizes, 95)),
        'edges_per_case_max':    float(ind_sizes.max()),
        'e_ind_total':           int(ind_sizes.sum()),
        'e_ind_over_ek':         float(ind_sizes.sum() / max(K, 1)),
        'ocr_b100':              float((ind_sizes > 100).mean()),
        'ocr_b500':              float((ind_sizes > 500).mean()),
        'ocr_b1000':             float((ind_sizes > 1000).mean()),
    }

def load_dataset_artifacts(artif_dir, probs_dir, model='GraphSAGE', seed=44, dataset_name=''):
    artif_dir = Path(artif_dir)
    probs_dir = Path(probs_dir)
    npz_path  = probs_dir / f'{model}_seed{seed}_test.npz'
    if not npz_path.exists():
        raise FileNotFoundError(f'Probs nao encontrado: {npz_path}')
    npz  = np.load(npz_path)
    p_te = np.asarray(npz['p'], dtype=float)
    y_te = np.asarray(npz['y'], dtype=int)
    print(f'[{dataset_name}] Scores: {len(p_te):,} arestas, {y_te.sum():,} positivas ({100*y_te.mean():.2f}%)')
    cache_path = artif_dir / 'edge_data_v4_clean.pt'
    if not cache_path.exists():
        raise FileNotFoundError(f'Cache nao encontrado: {cache_path}')
    cache  = torch.load(cache_path, map_location='cpu', weights_only=False)
    ei_all = cache['ei_all_cpu']
    te_idx = cache['te_idx']
    if isinstance(te_idx, torch.Tensor): te_idx = te_idx.numpy()
    if isinstance(ei_all, torch.Tensor): ei_all = ei_all.numpy()
    src_te = ei_all[0, te_idx]
    dst_te = ei_all[1, te_idx]
    assert len(p_te) == len(src_te), f'Mismatch: {len(p_te)} scores vs {len(src_te)} arestas'
    print(f'[{dataset_name}] Cache: {ei_all.shape[1]:,} arestas totais, {len(te_idx):,} no teste')
    ranked  = {'scores': p_te, 'y': y_te, 'src': src_te, 'dst': dst_te}
    full    = {'src': src_te, 'dst': dst_te}
    gt      = {'y_full': y_te}
    return ranked, full, gt, te_idx, ei_all

print('Funcoes base definidas.')

## Celula 3 -- B1, B2, B3

In [ ]:
# CELULA 3 -- Implementacoes de B1, B2, B3

# ---- B1: Temporal-WCC ----
# Algoritmo:
# 1. Selecionar top-k. 2. WCC padrao. 3. Para cada componente,
# ordenar arestas por timestamp e dividir quando gap > delta_steps.
def build_b1_temporal_wcc(ranked_edges, full_eval_edges, k=0.01, delta_steps=7, min_nodes=3):
    scores = np.asarray(ranked_edges['scores'], dtype=float)
    src    = np.asarray(ranked_edges['src'],    dtype=np.int64)
    dst    = np.asarray(ranked_edges['dst'],    dtype=np.int64)
    ts     = np.asarray(ranked_edges['timestamps'], dtype=np.int64)
    K   = max(1, int(math.ceil(k * len(scores))))
    sel = np.argsort(-scores)[:K]
    src_sel = src[sel]; dst_sel = dst[sel]
    # WCC padrao para obter componentes
    wcc_cases = _build_wcc(src_sel, dst_sel, sel, full_eval_edges, min_nodes=1)
    src_full = np.asarray(full_eval_edges['src'], dtype=np.int64)
    dst_full = np.asarray(full_eval_edges['dst'], dtype=np.int64)
    ts_full  = np.asarray(full_eval_edges['timestamps'], dtype=np.int64)
    final_cases = []
    for case in wcc_cases:
        seed_idxs = np.array(case['seed_edges'], dtype=np.int64)
        if len(seed_idxs) == 0: continue
        ts_case   = ts[seed_idxs]
        order     = np.argsort(ts_case)
        ts_sorted = ts_case[order]
        idx_sorted = seed_idxs[order]
        gaps   = np.diff(ts_sorted)
        breaks = np.where(gaps > delta_steps)[0] + 1
        splits = np.split(idx_sorted, breaks)
        for chunk in splits:
            if len(chunk) == 0: continue
            c_src = src[chunk]; c_dst = dst[chunk]
            node_set = set(c_src.tolist()) | set(c_dst.tolist())
            if len(node_set) < min_nodes: continue
            t_min = int(ts[chunk].min())
            t_max = int(ts[chunk].max())
            ind = [i for i, (u, v, t) in enumerate(zip(src_full, dst_full, ts_full))
                   if int(u) in node_set and int(v) in node_set and t_min <= t <= t_max]
            final_cases.append({'nodes': node_set, 'seed_edges': list(chunk),
                                 'induced_edges': ind, 't_min': t_min, 't_max': t_max})
    return final_cases

# ---- B2: WCC + Contexto Temporal ----
# Algoritmo:
# 1. WCC padrao. 2. Para cada caso, expandir janela de tempo por delta_ctx.
# 3. Adicionar arestas do periodo completo que caem na janela expandida
#    e envolvem ao menos um no do caso. 4. Recalcular arestas induzidas.
def build_b2_wcc_temporal_ctx(ranked_edges, full_eval_edges, k=0.01, delta_ctx=3, min_nodes=3):
    scores = np.asarray(ranked_edges['scores'], dtype=float)
    src    = np.asarray(ranked_edges['src'],    dtype=np.int64)
    dst    = np.asarray(ranked_edges['dst'],    dtype=np.int64)
    ts     = np.asarray(ranked_edges['timestamps'], dtype=np.int64)
    K   = max(1, int(math.ceil(k * len(scores))))
    sel = np.argsort(-scores)[:K]
    wcc_cases = _build_wcc(src[sel], dst[sel], sel, full_eval_edges, min_nodes=min_nodes)
    src_full = np.asarray(full_eval_edges['src'], dtype=np.int64)
    dst_full = np.asarray(full_eval_edges['dst'], dtype=np.int64)
    ts_full  = np.asarray(full_eval_edges['timestamps'], dtype=np.int64)
    final_cases = []
    for case in wcc_cases:
        seed_idxs = np.array(case['seed_edges'], dtype=np.int64)
        if len(seed_idxs) == 0: continue
        ts_seeds  = ts[seed_idxs]
        t_min = int(ts_seeds.min()) - delta_ctx
        t_max = int(ts_seeds.max()) + delta_ctx
        node_set = case['nodes']
        ctx_nodes = set()
        for u, v, t in zip(src_full, dst_full, ts_full):
            if t_min <= t <= t_max and (int(u) in node_set or int(v) in node_set):
                ctx_nodes.add(int(u)); ctx_nodes.add(int(v))
        expanded = node_set | ctx_nodes
        ind = [i for i, (u, v, t) in enumerate(zip(src_full, dst_full, ts_full))
               if int(u) in expanded and int(v) in expanded and t_min <= t <= t_max]
        final_cases.append({'nodes': expanded, 'seed_edges': list(seed_idxs),
                             'induced_edges': ind, 't_min': t_min, 't_max': t_max})
    return final_cases

# ---- B3: Hub-Pruned WCC ----
# Algoritmo:
# 1. Calcular grau no conjunto completo. 2. H = percentil hub_pct.
# 3. Remover do top-k arestas com nos de grau > H. 4. WCC sobre restante.
def build_b3_hub_pruned_wcc(ranked_edges, full_eval_edges, k=0.01, hub_pct=95, min_nodes=3):
    scores = np.asarray(ranked_edges['scores'], dtype=float)
    src    = np.asarray(ranked_edges['src'],    dtype=np.int64)
    dst    = np.asarray(ranked_edges['dst'],    dtype=np.int64)
    K      = max(1, int(math.ceil(k * len(scores))))
    sel    = np.argsort(-scores)[:K]
    src_full = np.asarray(full_eval_edges['src'], dtype=np.int64)
    dst_full = np.asarray(full_eval_edges['dst'], dtype=np.int64)
    max_node = int(max(src_full.max(), dst_full.max())) + 1
    degree   = np.zeros(max_node, dtype=np.int64)
    for u, v in zip(src_full, dst_full):
        degree[u] += 1; degree[v] += 1
    H       = np.percentile(degree[degree > 0], hub_pct)
    src_sel = src[sel]; dst_sel = dst[sel]
    is_hub  = (degree[src_sel] > H) | (degree[dst_sel] > H)
    mask    = ~is_hub
    n_rem   = int(is_hub.sum())
    print(f'    B3: H={H:.0f} | {n_rem}/{K} arestas hub removidas ({100*n_rem/K:.1f}%)')
    if mask.sum() == 0: return []
    return _build_wcc(src_sel[mask], dst_sel[mask], sel[mask], full_eval_edges, min_nodes=min_nodes)

print('B1, B2, B3 definidos.')

## Celula 4 -- Carregar dados e timestamps

In [ ]:
# CELULA 4 -- Carregar artefatos + timestamps do CSV original

def load_timestamps_from_csv(data_dir, te_idx, ei_all):
    data_dir   = Path(data_dir)
    candidates = [
        'transactions.csv','transaction.csv',
        'hi-large_trans.csv','hi-medium_trans.csv','hi-small_trans.csv',
        'li-large_trans.csv','li-medium_trans.csv','li-small_trans.csv',
    ]
    csv_path = None
    for c in candidates:
        found = list(data_dir.rglob(c))
        if found:
            csv_path = found[0]
            break
    if csv_path is None:
        raise FileNotFoundError(f'CSV de transacoes nao encontrado em {data_dir}')
    print(f'  CSV: {csv_path.name}')
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip().str.lower()
    time_col = next((c for c in ['timestamp','time','date','datetime','step'] if c in df.columns), None)
    if time_col is None:
        raise KeyError(f'Coluna de tempo nao encontrada. Colunas: {list(df.columns)}')
    print(f'  Coluna de tempo: {time_col!r}')
    if pd.api.types.is_numeric_dtype(df[time_col]):
        ts_all = pd.to_numeric(df[time_col], errors='coerce').fillna(0).astype(np.int64).values
    else:
        ts_all = (pd.to_datetime(df[time_col], errors='coerce').astype(np.int64) // 10**9).values
    assert len(ts_all) == ei_all.shape[1], (
        f'Mismatch: {len(ts_all)} timestamps vs {ei_all.shape[1]} arestas no cache.\n'
        f'Verifique se o CSV correto esta sendo carregado.'
    )
    ts_test = ts_all[te_idx]
    print(f'  Timestamps: min={ts_test.min()}, max={ts_test.max()}, range={ts_test.max()-ts_test.min()}')
    return ts_test

print('Carregando AML100k...')
ranked_100k, full_100k, gt_100k, te_idx_100k, ei_100k = load_dataset_artifacts(
    AML100K_ARTIF, AML100K_PROBS, model=AML100K_MODEL, seed=AML100K_SEED, dataset_name='AML100k')
print('  Carregando timestamps AML100k...')
ts_100k = load_timestamps_from_csv(AML100K_BASE, te_idx_100k, ei_100k)
ranked_100k['timestamps'] = ts_100k
full_100k['timestamps']   = ts_100k

print('\nCarregando AML1M (pode demorar ~1-2 min)...')
ranked_1m, full_1m, gt_1m, te_idx_1m, ei_1m = load_dataset_artifacts(
    AML1M_ARTIF, AML1M_PROBS, model=AML1M_MODEL, seed=AML1M_SEED, dataset_name='AML1M')
print('  Carregando timestamps AML1M...')
ts_1m = load_timestamps_from_csv(AML1M_BASE, te_idx_1m, ei_1m)
ranked_1m['timestamps'] = ts_1m
full_1m['timestamps']   = ts_1m

print('\nDados + timestamps carregados!')

## Celula 5 -- Hiperparametros

In [ ]:
# CELULA 5 -- Hiperparametros (tunar em val, nao no teste!)

B1_DELTA     = 7     # gap max entre arestas de um caso (steps). Sweep: {1,3,7,14,30}
B2_DELTA_CTX = 3     # janela de contexto alem do intervalo. Sweep: {1,3,7,14}
B3_HUB_PCT   = 95    # percentil de grau para hub. Sweep: {90,95,97,99}
KS           = [0.01, 0.02, 0.05, 0.10]
MIN_NODES    = 3

print(f'B1: delta={B1_DELTA} steps')
print(f'B2: delta_ctx={B2_DELTA_CTX} steps')
print(f'B3: hub_pct={B3_HUB_PCT}%')
print(f'k: {[str(int(k*100))+"%" for k in KS]}')

## Celula 6 -- Rodar B0 + B1 + B2 + B3

In [ ]:
# CELULA 6 -- Executar todos os metodos
DATASETS = [
    ('AML100k', ranked_100k, full_100k, gt_100k),
    ('AML1M',   ranked_1m,   full_1m,   gt_1m),
]

rows = []
for ds_name, ranked, full, gt in DATASETS:
    print(f'\n{"="*55}\n  Dataset: {ds_name}\n{"="*55}')
    for k in KS:
        kpct = f'{int(k*100)}%'
        print(f'\n  k={kpct}:')
        scores_arr = np.asarray(ranked['scores'])
        K_val = max(1, int(math.ceil(k * len(scores_arr))))
        sel_arr = np.argsort(-scores_arr)[:K_val]
        src_arr = np.asarray(ranked['src'])
        dst_arr = np.asarray(ranked['dst'])

        # B0
        with measure_resources() as res:
            cases = _build_wcc(src_arr[sel_arr], dst_arr[sel_arr], sel_arr, full, min_nodes=MIN_NODES)
            m = evaluate_cases(cases, gt, ranked, k)
        rows.append({'dataset':ds_name,'method':'B0_WCC','k%':kpct,'k_frac':k,**m,'time_s':res['time_s'],'ram_mb':res['ram_mb']})
        print(f"    B0 WCC:      #casos={m['n_cases']:,} | cov={m['coverage']:.3f} | purity={m['purity_induced']:.4f} | OCR(100)={m['ocr_b100']:.3f} | E_ind/E_k={m['e_ind_over_ek']:.2f}x | {res['time_s']:.1f}s")

        # B1
        with measure_resources() as res:
            cases = build_b1_temporal_wcc(ranked, full, k=k, delta_steps=B1_DELTA, min_nodes=MIN_NODES)
            m = evaluate_cases(cases, gt, ranked, k)
        rows.append({'dataset':ds_name,'method':'B1_TemporalWCC','k%':kpct,'k_frac':k,**m,'time_s':res['time_s'],'ram_mb':res['ram_mb'],'delta':B1_DELTA})
        print(f"    B1 TempWCC:  #casos={m['n_cases']:,} | cov={m['coverage']:.3f} | purity={m['purity_induced']:.4f} | OCR(100)={m['ocr_b100']:.3f} | E_ind/E_k={m['e_ind_over_ek']:.2f}x | {res['time_s']:.1f}s")

        # B2
        with measure_resources() as res:
            cases = build_b2_wcc_temporal_ctx(ranked, full, k=k, delta_ctx=B2_DELTA_CTX, min_nodes=MIN_NODES)
            m = evaluate_cases(cases, gt, ranked, k)
        rows.append({'dataset':ds_name,'method':'B2_WCC_Ctx','k%':kpct,'k_frac':k,**m,'time_s':res['time_s'],'ram_mb':res['ram_mb'],'delta':B2_DELTA_CTX})
        print(f"    B2 WCC+Ctx:  #casos={m['n_cases']:,} | cov={m['coverage']:.3f} | purity={m['purity_induced']:.4f} | OCR(100)={m['ocr_b100']:.3f} | E_ind/E_k={m['e_ind_over_ek']:.2f}x | {res['time_s']:.1f}s")

        # B3
        with measure_resources() as res:
            cases = build_b3_hub_pruned_wcc(ranked, full, k=k, hub_pct=B3_HUB_PCT, min_nodes=MIN_NODES)
            m = evaluate_cases(cases, gt, ranked, k)
        rows.append({'dataset':ds_name,'method':'B3_HubPruned','k%':kpct,'k_frac':k,**m,'time_s':res['time_s'],'ram_mb':res['ram_mb'],'hub_pct':B3_HUB_PCT})
        print(f"    B3 HubPrune: #casos={m['n_cases']:,} | cov={m['coverage']:.3f} | purity={m['purity_induced']:.4f} | OCR(100)={m['ocr_b100']:.3f} | E_ind/E_k={m['e_ind_over_ek']:.2f}x | {res['time_s']:.1f}s")

df_results = pd.DataFrame(rows)
print('\nTodos os metodos concluidos!')

## Celula 7 -- Tabela comparativa e analise go/no-go

In [ ]:
# CELULA 7 -- Tabela e analise go/no-go
for ds_name in ['AML100k', 'AML1M']:
    df_ds = df_results[df_results['dataset'] == ds_name].copy()
    pivot = df_ds.pivot_table(
        index='k%', columns='method',
        values=['n_cases','coverage','purity_induced','ocr_b100','e_ind_over_ek','time_s'],
        aggfunc='first'
    ).round(4)
    print(f'\n=== {ds_name} ===')
    display(pivot)

print('\n' + '='*60)
print('ANALISE GO / NO-GO -- B1 vs B0 (AML1M, k=5% e k=10%)')
print('='*60)
print('Criterio: B1 deve reduzir OCR-B(100) em >= 70% para ser')
print('          um baseline forte que mude a narrativa.')

go_signal = False
for kpct in ['5%', '10%']:
    b0 = df_results[(df_results['dataset']=='AML1M') & (df_results['method']=='B0_WCC') & (df_results['k%']==kpct)]
    b1 = df_results[(df_results['dataset']=='AML1M') & (df_results['method']=='B1_TemporalWCC') & (df_results['k%']==kpct)]
    if b0.empty or b1.empty: continue
    ocr_b0 = float(b0.iloc[0]['ocr_b100'])
    ocr_b1 = float(b1.iloc[0]['ocr_b100'])
    reducao = (ocr_b0 - ocr_b1) / max(ocr_b0, 1e-9)
    cov_b1  = float(b1.iloc[0]['coverage'])
    cov_b0  = float(b0.iloc[0]['coverage'])
    status  = 'GO' if reducao >= 0.70 else ('PARCIAL' if reducao >= 0.40 else 'NO-GO')
    if reducao >= 0.70: go_signal = True
    print(f'  k={kpct}: OCR(B0)={ocr_b0:.3f} -> OCR(B1)={ocr_b1:.3f}  '
          f'reducao={100*reducao:.1f}%  |  perda_cov={cov_b0-cov_b1:+.3f}  [{status}]')

print()
if go_signal:
    print('-> B1 resolve parte relevante. Avaliar mudanca de narrativa.')
else:
    print('-> Temporal puro nao resolve. BTCS precisa de Lk + Leiden.')
    print('   Narrativa permanece: BTCS vs WCC baseline.')

## Celula 8 -- Plot OCR-B por metodo

In [ ]:
# CELULA 8 -- Plot comparativo
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
METHODS = ['B0_WCC', 'B1_TemporalWCC', 'B2_WCC_Ctx', 'B3_HubPruned']
COLORS  = {'B0_WCC':'#1f77b4','B1_TemporalWCC':'#ff7f0e','B2_WCC_Ctx':'#2ca02c','B3_HubPruned':'#d62728'}
LABELS  = {'B0_WCC':'B0 WCC','B1_TemporalWCC':'B1 Temporal','B2_WCC_Ctx':'B2 WCC+Ctx','B3_HubPruned':'B3 Hub-Pruned'}

for ax, ds in zip(axes, ['AML100k', 'AML1M']):
    df_ds = df_results[df_results['dataset'] == ds]
    x = np.arange(len(KS))
    w = 0.18
    for j, m in enumerate(METHODS):
        df_m = df_ds[df_ds['method'] == m]
        if df_m.empty: continue
        vals = []
        for k in KS:
            row = df_m[df_m['k%'] == f'{int(k*100)}%']
            vals.append(float(row['ocr_b100'].values[0]) if not row.empty else 0)
        ax.bar(x + j*w, vals, w, label=LABELS[m], color=COLORS[m], alpha=0.85)
    ax.set_title(f'{ds} -- OCR(B=100)')
    ax.set_xlabel('k%')
    ax.set_ylabel('Fracao de casos com >100 arestas')
    ax.set_xticks(x + 1.5*w)
    ax.set_xticklabels([f'{int(k*100)}%' for k in KS])
    ax.legend(fontsize=8)

plt.tight_layout()
plot_path = BTCS_OUTPUT_DIR / 'b0_b1_b2_b3_ocr_comparison.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Plot salvo: {plot_path}')

## Celula 9 -- Export

In [ ]:
# CELULA 9 -- Salvar resultados
out_csv = BTCS_OUTPUT_DIR / 'b0_b1_b2_b3_results.csv'
df_results.to_csv(out_csv, index=False)
print(f'CSV salvo: {out_csv}')

latex_cols = ['dataset','method','k%','n_cases','coverage','purity_induced','cr_at_k','e_ind_over_ek','ocr_b100','time_s']
out_tex = BTCS_OUTPUT_DIR / 'b0_b1_b2_b3_table.tex'
df_results[latex_cols].round(4).to_latex(out_tex, index=False, escape=False)
print(f'LaTeX salvo: {out_tex}')

print('\nRoadmap:')
print('1. [ok] nb00 -- WCC reproduzido')
print('2. [ok] nb01 -- B1, B2, B3 (este notebook)')
print('3. [ ] nb02 -- Grafo Lk + Leiden + Contextualizacao (BTCS)')
print('4. [ ] nb03 -- Ablacoes sistematicas')
print('5. [ ] nb04 -- AMLworld + Libra Bank')